In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
data = fetch_california_housing()

x_train_full,x_test,y_train_full,y_test =  train_test_split(data.data, data.target, random_state=42)  # 75 : 25
x_train,x_val,y_train,y_val = train_test_split(x_train_full,y_train_full,random_state=42)
x_train.shape, x_val.shape, x_test.shape


- 훈련데이터 가중치를 학습
- 미니배치 경사하강법 + 역전파
- 손실을 줄이도록 파라메터를 반복 업데이트


In [ ]:
import tensorflow as tf

In [ ]:
x_train.shape[1:]

In [ ]:
tf.random.set_seed(42)
normal_layer = tf.keras.layers.Normalization(input_shape=(8,))
model = tf.keras.Sequential([
    normal_layer,
    tf.keras.layers.Dense(50,activation='relu'),
    tf.keras.layers.Dense(50,activation='relu'),
    tf.keras.layers.Dense(50,activation='relu'),
    tf.keras.layers.Dense(1)
])
model.compile(loss = 'mse', optimizer='adam', metrics=['RootMeanSquaredError'])
normal_layer.adapt(x_train)
history = model.fit(x_train, y_train ,epochs=50, validation_data=(x_val,y_val))

In [ ]:
# 성능측정
model.evaluate(x_test, y_test)

In [ ]:
history.history.keys()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(range(50), history.history['RootMeanSquaredError'], label = 'RootMeanSquaredError')
ax.plot(range(50), history.history['loss'], label = 'loss')
ax.plot(range(50), history.history['val_RootMeanSquaredError'], label='val_RootMeanSquaredError')
ax.plot(range(50), history.history['val_loss'], label='val_loss')
ax.legend()
plt.show()

In [ ]:
from sklearn.metrics import r2_score
y_predcit = model.predict(x_test)
r2_score(y_test, y_predcit)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
sk_model = RandomForestRegressor(random_state=42)
sk_model.fit(x_train,y_train)
sk_y_predict = sk_model.predict(x_test)
r2_score(y_test, sk_y_predict)

함수형 API를 사용한 복잡한 모델 구축
 - 와이드 & 딥 신경망 : 입력의 전부 또는 일부를 출력레이어 직접연결
 - 선형관계와 비선형표현을 동시에 학습해 편향/분산 의 균형을 맞춤
 - 저차원과 고차원 정보를 합쳐서 정보손실을 줄임

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
data = fetch_california_housing()

x_train_full,x_test,y_train_full,y_test =  train_test_split(data.data, data.target, random_state=42)  # 75 : 25
x_train,x_val,y_train,y_val = train_test_split(x_train_full,y_train_full,random_state=42)
x_train.shape, x_val.shape, x_test.shape


import tensorflow as tf
tf.keras.backend.clear_session()
tf.random.set_seed(42)
input = tf.keras.layers.Input(shape=x_train.shape[1:])
normalization_layer  =tf.keras.layers.Normalization()
hidden_layer1 = tf.keras.layers.Dense(30,activation='relu')
output_layer = tf.keras.layers.Dense(1)
concat_layer = tf.keras.layers.Concatenate()

normalized = normalization_layer(input)
hidden1 = hidden_layer1(normalized)
concat =  concat_layer([normalized, hidden1])
output = output_layer(concat)

model = tf.keras.Model(inputs = [input], outputs= [output])

model.summary()  # 파라메터 개수 확인 ?? 데이터수에 비해 과도하면 과적합 위험

In [ ]:
model.compile(loss = 'mse', optimizer='adam', metrics=['RootMeanSquaredError'])
normalization_layer.adapt(x_train)
epochs = 30
history = model.fit(x_train, y_train ,epochs=epochs, validation_data=(x_val,y_val))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(range(epochs), history.history['RootMeanSquaredError'], label = 'RootMeanSquaredError')
ax.plot(range(epochs), history.history['loss'], label = 'loss')
ax.plot(range(epochs), history.history['val_RootMeanSquaredError'], label='val_RootMeanSquaredError')
ax.plot(range(epochs), history.history['val_loss'], label='val_loss')
ax.legend()
plt.show()

In [ ]:
from sklearn.metrics import r2_score
y_predcit = model.predict(x_test)
r2_score(y_test, y_predcit)

다중경로

In [ ]:
from sklearn.base import validate_data
x_train_wide,x_train_deep =  x_train[:, :5], x_train[:,2:]
x_val_wide,x_val_deep =  x_val[:, :5], x_val[:,2:]
x_test_wide,x_test_deep =  x_test[:, :5], x_test[:,2:]


input_wide = tf.keras.layers.Input(shape=[5])  # 0 ~ 4 특성
input_deep = tf.keras.layers.Input(shape=[6])  # 2 ~ 7
norm_layer_wide = tf.keras.layers.Normalization()
norm_layer_deep = tf.keras.layers.Normalization()
norm_wide = norm_layer_wide(input_wide)
norm_deep = norm_layer_deep(input_deep)

hidden1 = tf.keras.layers.Dense(30, activation='relu')( norm_deep )
hidden2 = tf.keras.layers.Dense(30, activation='relu')( hidden1 )

concat = tf.keras.layers.concatenate([norm_wide, hidden2])
output = tf.keras.layers.Dense(1)(concat)

aux_output = tf.keras.layers.Dense(1)(hidden2)

model = tf.keras.Model(inputs = [input_wide, input_deep], outputs = [output, aux_output])

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss =('mse','mse'), loss_weights=(0.9,0.1) , optimizer=optimizer,
              metrics=['RootMeanSquaredError','RootMeanSquaredError'])

norm_layer_wide.adapt(x_train_wide)
norm_layer_deep.adapt(x_train_deep)
history = model.fit(
    (x_train_wide, x_train_deep), (y_train,y_train), epochs=20,
    validation_data = ( (x_val_wide, x_val_deep), (y_val, y_val)     )
)

In [ ]:
model.summary()

In [ ]:
model.evaluate( (x_test_wide, x_test_deep),(y_test,y_test) )

In [ ]:
model.predict( (x_test_wide, x_test_deep) )

In [ ]:
y_test